In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import re
import secrets
import string
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Core Logic Functions
def analyze_password(password):
    """Evaluates password strength metrics."""
    score = 0
    feedback = []
    
    # Mock Database: History of leaked/previously used passwords
    LEAKED_HISTORY = ["Password123!", "Admin@2024", "LetMeIn__2025", "Qwerty!@#"]
    
    # Check 1: Length
    length = len(password)
    if length >= 12:
        score += 2
    elif length >= 8:
        score += 1
        feedback.append("⚠️ Password is short. Aim for 12+ characters to increase complexity.")
    else:
        return {"score": 0, "status": "Very Weak", "feedback": ["❌ Critical: Must be at least 8 characters long."]}

    # Check 2: Character Diversity (Complexity)
    if re.search(r"[a-z]", password): score += 1
    else: feedback.append("💡 Add lowercase letters.")
        
    if re.search(r"[A-Z]", password): score += 1
    else: feedback.append("💡 Add uppercase letters.")
        
    if re.search(r"\d", password): score += 1
    else: feedback.append("💡 Add numbers.")
        
    if re.search(r"[ !@#$%^&*(),.?\":{}|<>_+\-=\[\]\\/;`~]", password): score += 1
    else: feedback.append("💡 Add special characters (e.g., !, @, #).")

    # Check 3: Uniqueness / History Match
    if password in LEAKED_HISTORY:
        return {"score": 0, "status": "Compromised", "feedback": ["❌ Security Risk: Matches a known or previously used password!"]}

    # Grading
    status = "Strong" if score >= 5 else "Medium" if score >= 3 else "Weak"
    return {"score": score, "status": status, "feedback": feedback}


def generate_secure_alternative(length=14):
    """Generates a cryptographically secure alternative."""
    pool = string.ascii_letters + string.digits + "!@#$%^&*()_+"
    while True:
        pwd = ''.join(secrets.choice(pool) for _ in range(length))
        if (any(c.islower() for c in pwd) and any(c.isupper() for c in pwd)
                and any(c.isdigit() for c in pwd) and any(c in "!@#$%^&*()_+" for c in pwd)):
            return pwd

# 2. UI Layout Components
title_html = widgets.HTML("<h2>🛡️ Kaggle Password Strength Analyzer & Generator</h2><hr>")
input_box = widgets.Password(description='Password:', placeholder='Type here...', style={'description_width': 'initial'})
validate_btn = widgets.Button(description='Validate Password', button_style='primary', icon='shield')
generate_btn = widgets.Button(description='Suggest Alternative', button_style='success', icon='key')
output_panel = widgets.Output()

# 3. Widget Event Handlers
def handle_validation(b):
    with output_panel:
        clear_output()
        pwd = input_box.value.strip()
        if not pwd:
            print("Please input a password to evaluate.")
            return
        
        res = analyze_password(pwd)
        print(f"Testing Profile: {'*' * len(pwd)}")
        print(f"Security Rating: **{res['status']}** ({res['score']}/6 Points)")
        print("-" * 40)
        if res['feedback']:
            for line in res['feedback']:
                print(line)
        else:
            print("✅ Excellent! This password satisfies modern cryptographic criteria.")

def handle_generation(b):
    with output_panel:
        clear_output()
        secure_option = generate_secure_alternative()
        print("💡 Secure Alternative Generated via CSPRNG:")
        print(f"👉 ** {secure_option} **")
        print("\n*Tip: This password was compiled utilizing OS-level randomness, ensuring extreme resistance to brute-force algorithms.*")

validate_btn.on_click(handle_validation)
generate_btn.on_click(handle_generation)

# 4. Render
display(title_html, input_box, widgets.HBox([validate_btn, generate_btn]), output_panel)

HTML(value='<h2>🛡️ Kaggle Password Strength Analyzer & Generator</h2><hr>')

Password(description='Password:', placeholder='Type here...', style=TextStyle(description_width='initial'))

Output()

In [3]:
import pandas as pd

print("--- Transforming Password Metrics into Machine Learning Features ---")

# Simulated Raw Dataset
raw_data = {
    'user_id': [101, 102, 103, 104, 105],
    'raw_password': ['password', 'Admin@2024', 'qwerty123456789', 'K@ggle_M0del_2026', '123456']
}
df = pd.DataFrame(raw_data)

# Pipeline Application
df['metrics'] = df['raw_password'].apply(analyze_password)

# Feature Extraction/Engineering
df['pwd_length'] = df['raw_password'].apply(len)
df['score'] = df['metrics'].apply(lambda x: x['score'])
df['security_tier'] = df['metrics'].apply(lambda x: x['status'])
df['is_compromised'] = df['security_tier'].apply(lambda x: 1 if x == "Compromised" else 0)

# Drop raw dictionary column for clean viewing
df_clean = df.drop(columns=['metrics'])

# Display Output Table
display(df_clean)

--- Transforming Password Metrics into Machine Learning Features ---


,user_id,raw_password,pwd_length,score,security_tier,is_compromised
0,101,password,8,2,Weak,0
1,102,Admin@2024,10,0,Compromised,1
2,103,qwerty123456789,15,4,Medium,0
3,104,K@ggle_M0del_2026,17,6,Strong,0
4,105,123456,6,0,Very Weak,0
